In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

import pandas as pd
pd.set_option("display.max_columns",30)
import requests

In [ ]:

"""
Steps for the Transform Load Lambda function
    
1. Get data from S3 (taxi, weather)
2. Weather data transformations
3. Taxi data transformations - DONE
4. Update dim_payment_type
5. Update dim_company
6. Update fact_taxi_trips with the ids from dim_payment_type and dim_company
7. Upload dim_weather to S3
8. Upload fact_taxi_trips to S3
9. Upload dim_payment_type and dim_company (current, and previous version)

"""


'\nSteps for the Transform Load Lambda function\n\n1. Get data from S3 (taxi, weather)\n2. Weather data transformations\n3. Taxi data transformations\n4. Update dim_payment_type\n5. Update dim_company\n6. Update fact_taxi_trips with the ids from dim_payment_type and dim_company\n7. Upload dim_weather to S3\n8. Upload fact_taxi_trips to S3\n9. Upload dim_payment_type and dim_company (current, and previous version)\n\n'

In [3]:
current_datetime = datetime.now() - relativedelta(month=2)
formatted_datetime = current_datetime.strftime("%Y-%m-%d")

url = (

    f"https://data.cityofchicago.org/resource/ajtu-isnz.json?"
    f"$where=trip_start_timestamp >= '{formatted_datetime}T00:00:00' "
    f"AND trip_start_timestamp <= '{formatted_datetime}T23:59:59' "
    f"&$limit=30000"
)

response = requests.get(url)
data = response.json()

taxi_trips = pd.DataFrame(data)

taxi_trips.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,000374dcbfb7ccf6e0abe3b0021e1da12786aa58,179f1a051e9e6d3fc0726628962faff68506086ee8df14...,2026-02-24T23:45:00.000,2026-02-25T00:00:00.000,1532,14.84,76,7,37.5,8.4,0,4,50.4,Credit Card,Blue Ribbon Taxi Association,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.922686284,-87.649488729,"{'type': 'Point', 'coordinates': [-87.64948872...",NaN,NaN
1,f47ffeda3b022b9694095aade874328d8b55f2cb,fe43642f2a2fb98dc45bea2b95638dc0c15612e66f229c...,2026-02-24T23:45:00.000,2026-02-25T00:30:00.000,2607,22.62,76,1,57,2,0,4,63.5,Credit Card,5 Star Taxi,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",42.009622881,-87.670166857,"{'type': 'Point', 'coordinates': [-87.67016685...",NaN,NaN
2,e4d07301a6705c1758eb0cb43abc9476c013b2c5,a7aaa6374b9f88b5fd31a1106378dffccb77e1261bc57a...,2026-02-24T23:45:00.000,2026-02-25T00:00:00.000,1328,10.46,14,34,28.5,0,0,0,28.5,Prcard,5 Star Taxi,41.968069,-87.721559063,"{'type': 'Point', 'coordinates': [-87.72155906...",41.842076117,-87.633973422,"{'type': 'Point', 'coordinates': [-87.63397342...",NaN,NaN
3,e3b4fdb3af0276e16f4c8f5ebfac53be88b7ea97,a692055914b912499c7042ab77631e9da33065435edbde...,2026-02-24T23:45:00.000,2026-02-25T00:00:00.000,922,11.58,24,10,29.75,0,0,0,29.75,Prcard,Flash Cab,41.901206994,-87.676355989,"{'type': 'Point', 'coordinates': [-87.67635598...",41.985015101,-87.804532006,"{'type': 'Point', 'coordinates': [-87.80453200...",NaN,NaN
4,e250417b60245c8ae60c5ea54314072f69818046,0734b10e5be7c6aa0f018e381095204a81011ef78493ba...,2026-02-24T23:45:00.000,2026-02-25T00:15:00.000,1588,14.12,76,3,36,10.12,0,4,50.62,Credit Card,City Service,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.96581197,-87.655878786,"{'type': 'Point', 'coordinates': [-87.65587878...",NaN,NaN


### Taxi data transformations

In [ ]:
def taxi_trips_transformations(taxi_trips: pd.DataFrame) -> pd.DataFrame:
    
    """Perform transformations on the taxi data

    1. Drop selected columns.
    2. Drop NULL values across all columns.
    3. Rename selected columns.
    4. Create "datetime_for_weather" helper column(for dim_weather join).
    
    :param taxi_trips:  The DataFrame holding the daily taxi trips.
    :raise TypeError:   When taxi_trips parameter is not a valid pandas DataFrame.
    :return:            Transformed taxi trips DataFrame.
    
    """
    if not isinstance (taxi_trips, pd.DataFrame):
        raise TypeError("taxi_trips is not a valid pandas DtaFrame.")
    taxi_trips.drop(["pickup_census_tract","dropoff_census_tract",
                    "pickup_centroid_location","dropoff_centroid_location"],axis=1,inplace=True)

    taxi_trips.dropna(inplace=True)

    taxi_trips.rename(columns= { "pickup_community_area" : "pickup_community_area_id", 
                            "dropoff_community_area" : "dropoff_community_area_id"
                            }, inplace=True)

    taxi_trips["trip_start_timestamp"] = pd.to_datetime(taxi_trips["trip_start_timestamp"])

    taxi_trips["datetime_for_weather"] = taxi_trips["trip_start_timestamp"].dt.floor("h")

    return taxi_trips

In [ ]:
taxi_trips_transformed = taxi_trips_transformations(taxi_trips)

taxi_trips_transformed.head()


KeyError: "['pickup_census_tract', 'dropoff_census_tract', 'pickup_centroid_location', 'dropoff_centroid_location'] not found in axis"

In [7]:
taxi_trips_transformed.info()

<class 'pandas.DataFrame'>
Index: 15468 entries, 0 to 17102
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     15468 non-null  str           
 1   taxi_id                     15468 non-null  str           
 2   trip_start_timestamp        15468 non-null  datetime64[us]
 3   trip_end_timestamp          15468 non-null  str           
 4   trip_seconds                15468 non-null  str           
 5   trip_miles                  15468 non-null  str           
 6   pickup_community_area_id    15468 non-null  str           
 7   dropoff_community_area_id   15468 non-null  str           
 8   fare                        15468 non-null  str           
 9   tips                        15468 non-null  str           
 10  tolls                       15468 non-null  str           
 11  extras                      15468 non-null  str           
 12  trip_t